# מעבדה 16 — קרינה תרמית וקיבול החום של מוצקים

במעבדה הזו אתם בונים מחדש את חוק פלאנק מהחלקים שהמודול גוזר — המתנד של מודול 12, וספירה של
הגלים העומדים בקופסה — ואז בודקים אותו. אתם סופרים את האופנים של חלל אחד אחד; מוצאים היכן
הספקטרום מגיע לשיא, ליחידת תדירות וליחידת אורך גל, ומגלים ששתי התשובות הן פוטונים שונים;
מבצעים אינטגרציה נומרית של הספקטרום וצופים בחוק $T^4$ ובקבוע שלו מתגלים; ואז פונים למוצקים,
שבהם אותה נוסחה, מסוכמת על פני התנודות של גביש, מותאמת לקיבול החום הנמדד של כסף
מ-$1\ \mathrm{K}$ עד $300\ \mathrm{K}$. אתם מסיימים בהתאמה לקרינת הרקע הקוסמית, כפי שנמדדה
על ידי הלוויין COBE של NASA, ובציטוט של שתי מדידות עם אי-הוודאויות שלהן מוצהרות ביושר.

המסלול: האובייקטים (חלק 1); אופן אחד, בנוי מחדש ממודול 12 (חלק 2); ספירת האופנים של חלל
(חלק 3); חוק פלאנק כמכפלה (חלק 4); שני ספקטרומים, שני שיאים (חלק 5); חוק סטפן–בולצמן
באינטגרציה נומרית (חלק 6); כסף בטמפרטורה נמוכה (חלק 7); איינשטיין מול דבאי (חלק 8); קרינת
הרקע הקוסמית (חלק 9); הבדיקות האוטומטיות (חלק 10); ושני כלי חקירה (חלק 11).

עברו עליה בסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את ניבויכם בתא המיועד לכך **לפני**
הרצת התא הבא. ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

**יחידות.** יחידות SI לכל אורך הדרך: תדירויות בהרץ, אורכי גל במטרים, ספקטרומים
ב-$\mathrm{J\,s\,m^{-3}}$ (ליחידת תדירות) או ב-$\mathrm{J\,m^{-4}}$ (ליחידת אורך גל), קיבולי
חום ב-$\mathrm{J\,K^{-1}}$ — למול, כאשר מספר האטומים הוא $N_A$.

## מפרט המודל

| | |
|---|---|
| **מערכת** | השדה האלקטרומגנטי בחלל בנפח $V$ בטמפרטורה $T$, כאופני גל עומד בלתי תלויים שאנרגיותיהם הן כפולות שלמות של $h\nu$; גביש של $N$ אטומים כ-$3N$ מתנדים כאלה, בתדירות אחת (איינשטיין) או מפוזרים על פני ספקטרום קולי קטוע (דבאי) |
| **דינמיקה** | אין — כל עקומה היא שיווי משקל תרמי |
| **גבול** | דפנות מחזירות לחלוטין המוחזקות ב-$T$; חור קטן פולט שטף השווה ל-$c/4$ כפול צפיפות האנרגיה |
| **צבר** | קנוני, אופן אחר אופן; מספר הפוטונים אינו נשמר, ולכן $\mu = 0$ |
| **מוזנח** | אנרגיית נקודת האפס, אינטראקציה בין פוטונים, מקדם פליטה קטן מאחד; במוצקים, נפיצת הפונונים הממשית, התפשטות תרמית ($C_P - C_V$) ואלקטרוני ההולכה (עד שחלק 7 מודד אותם) |
| **תקף כאשר** | החלל גדול בהרבה מ-$hc/(\kB T)$; הגביש הרמוני וחם דיו כדי שה-$\gamma T$ של האלקטרונים יהיה זניח |
| **אופני כישלון** | חללים זעירים או קרים מאוד; אור לא תרמי; מתכות מתחת לקלווינים ספורים; גבישים סמוך להתכה |

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from thermolab import partition, radiation
from thermolab.constants import K_B, N_A, R_GAS
from thermolab.validation import relative_error, scaling_exponent

H, C = radiation.PLANCK_H, radiation.C_LIGHT

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    DATA = Path("..", "..", "..", "data")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSVs co-located here.
    DATA = Path("data")

print(f"Wien roots: x = {radiation.WIEN_X_NU:.6f} (per frequency), "
      f"{radiation.WIEN_X_LAMBDA:.6f} (per wavelength)")
print(f"a = {radiation.RADIATION_CONSTANT:.4e} J m^-3 K^-4,  "
      f"sigma = {radiation.STEFAN_BOLTZMANN:.4e} W m^-2 K^-4")

## חלק 1 — האובייקטים

| פונקציה | מה היא נותנת |
|---|---|
| `mode_occupation(quantum, T)`, `mean_mode_energy(nu, T)` | $\langle n \rangle = 1/(e^{x} - 1)$ ו-$h\nu\langle n \rangle$ לאופן יחיד, $x = h\nu/(\kB T)$ |
| `mode_density(nu, V)`, `count_cavity_modes(L, nu_max)`, `mode_count_smooth(L, nu_max)` | $g(\nu) = 8\pi V\nu^2/c^3$; האופנים של קובייה, נספרים אחד אחד; הספירה החלקה $8\pi V\nu^3/(3c^3)$ |
| `planck_u_nu(nu, T)`, `planck_u_lambda(lam, T)`, `rayleigh_jeans_u_nu(nu, T)` | שתי הצפיפויות הספקטרליות, והצפיפות הקלאסית |
| `wien_peak_nu(T)`, `wien_peak_lambda(T)` | היכן כל ספקטרום מגיע לשיא |
| `integrate_u_nu(T, n)`, `integrate_u_lambda(T, n)`, `energy_density(T)`, `simpson(y, x)` | האנרגיה הכוללת באינטגרציה נומרית בכל אחד משני המשתנים, ו-$aT^4$ |
| `photon_gas(T)`, `photon_number_density(T)` | $u$, $P$, $s$ ו-$n$ של גז הפוטונים |
| `einstein_heat_capacity(T, theta_E, N)`, `debye_heat_capacity(T, theta_D, N)`, `debye_low_t(...)` | שני המודלים של מוצק, וחוק $T^3$ של דבאי |
| `fit_einstein_temperature`, `fit_debye_temperature`, `fit_blackbody_temperature` | התאמות ריבועים פחותים בפרמטר אחד; כל אחת מחזירה `.value`, `.sigma`, `.residuals`, `.rms_residual`, `.chi_squared` |

### לנבא

התחייבו לתשובה לכל אחת מהשאלות לפני שתריצו משהו.

1. אופן עם $h\nu = 3\kB T$: האם הוא מחזיק בערך $\kB T$, בערך $0.15\,\kB T$, או כלום?
2. ב-$5772\ \mathrm{K}$ הספקטרום ליחידת אורך גל מגיע לשיא ב-$502\ \mathrm{nm}$. ליחידת תדירות,
   האם הוא מגיע לשיא בתדירות של אור של $502\ \mathrm{nm}$?
3. חימום מ-$300$ ל-$600\ \mathrm{K}$: פי כמה גדלה האנרגיה הכוללת?
4. קיבול החום של כסף בין $1$ ל-$5\ \mathrm{K}$: איזה שיפוע יהיה לו בצירים לוגריתמיים כפולים?

**הניבויים שלכם:**

1.
2.
3.
4.

## חלק 2 — אופן אחד, בנוי מחדש ממודול 12

מודול 12 סכם את גורמי בולצמן של רמות המתנד. כאן הסכום המפורש הזה, המורץ על פני $4000$ רמות
$n\,h\nu$, מועמד מול הביטוי הסגור $\langle n \rangle = 1/(e^x - 1)$, והאנרגיה הממוצעת משורטטת
כנגד הערך הקלאסי $\kB T$.

**הניבוי שלכם:** באיזה ערך של $h\nu/(\kB T)$ יורדת האנרגיה הממוצעת למחצית מ-$\kB T$?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
quantum = 4e-21  # J
for temperature in (30.0, 300.0, 3000.0):
    levels = quantum * np.arange(4000)
    brute = partition.level_sums(levels, temperature).energy / quantum
    closed = radiation.mode_occupation(quantum, temperature)
    print(f"T = {temperature:6.0f} K: explicit sum <n> = {brute:.12f}, "
          f"closed form {closed:.12f}, relative difference {relative_error(closed, brute):.1e}")

x = np.linspace(0.01, 8, 400)
t_ref = 300.0
energy = radiation.mean_mode_energy(x * K_B * t_ref / H, t_ref) / (K_B * t_ref)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, energy, color="#2563eb", lw=2, label="quantum: x / (e^x - 1)")
ax.axhline(1.0, color="#dc2626", ls="--", label="classical: k_B T")
ax.set_xlabel("h nu / (k_B T)")
ax.set_ylabel("<E> / (k_B T)")
ax.legend()
plt.tight_layout()
plt.show()
half = x[np.argmin(np.abs(energy - 0.5))]
print(f"<E> falls to k_B T / 2 at h nu = {half:.2f} k_B T; "
      f"at 3 k_B T it is {3 / np.expm1(3):.3f} k_B T")

הביטוי הסגור הוא הסכום של מודול 12, עד שתים-עשרה ספרות. קוונט זול ($x \ll 1$) מחזיק את
הערך הקלאסי $\kB T$; קוונט יקר מחזיק כמות קטנה מעריכית. זה כל הקלט הקוונטי.

## חלק 3 — ספירת האופנים של חלל

לגל עומד בקובייה שאורך צלעה $L$ יש $\mathbf{k} = (\pi/L)(n_x, n_y, n_z)$ עם מספרים שלמים
חיוביים, ושני קיטובים. `count_cavity_modes` מונה את שלשות המספרים השלמים אחת אחת; הגזירה
החליפה אותן בנפח של אוקטנט. כמה טובה ההחלפה הזו, וכיצד היא משתפרת?

In [ ]:
side = 1.0
print(" radius |n|   explicit count     smooth count   shortfall")
shortfalls = {}
for radius in (5, 10, 20, 40, 80, 160, 320):
    nu = radius * C / (2 * side)
    counted = radiation.count_cavity_modes(side, nu)
    smooth = radiation.mode_count_smooth(side, nu)
    shortfalls[radius] = 1 - counted / smooth
    print(f"{radius:10d} {counted:16d} {smooth:16.0f}   {100 * shortfalls[radius]:6.2f} %")
print(f"shortfall x radius: {[round(r * s, 2) for r, s in shortfalls.items()]}")
print(f"a 1 cm cavity at 300 K: |n| near the peak = "
      f"{2 * 0.01 * radiation.wien_peak_nu(300.0) / C:.0f}")

מכפלת החוסר ברדיוס כמעט קבועה: השגיאה היא אפקט של פני השטח, שיורד כמו
$1/|\mathbf{n}|$. לכל חלל שתוכלו לבנות, בכל טמפרטורה שתוכלו להגיע אליה, הספירה החלקה מדויקת
לכל צורך מעשי.

## חלק 4 — חוק פלאנק כמכפלה

הכפילו את מספר האופנים ליחידת נפח וליחידת תדירות, $g(\nu)/V$, באנרגיה הממוצעת לאופן. עשו זאת
פעם אחת עם האנרגיה הממוצעת הקוונטית ופעם אחת עם הערך הקלאסי $\kB T$, והשוו ל-`planck_u_nu`.

**הניבוי שלכם:** היכן העקומות הקלאסיות והקוונטיות מסכימות, והיכן הן נפרדות?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colours = {1500.0: "#d97706", 3000.0: "#dc2626", 6000.0: "#2563eb"}
for temperature, colour in colours.items():
    nu = np.linspace(0, 3.5 * radiation.wien_peak_nu(6000.0), 800)
    product = radiation.mode_density(nu) * radiation.mean_mode_energy(nu, temperature)
    assert np.allclose(product, radiation.planck_u_nu(nu, temperature), rtol=1e-14, atol=0)
    ax.plot(nu / 1e12, product, color=colour, lw=2, label=f"Planck, {temperature:.0f} K")
    ax.plot(nu / 1e12, radiation.rayleigh_jeans_u_nu(nu, temperature), ls="--", color=colour,
            lw=1, label=f"classical, {temperature:.0f} K")
ax.axvspan(C / 750e-9 / 1e12, C / 380e-9 / 1e12, color="#fde68a", alpha=0.4, label="visible")
ax.set_ylim(0, 1.3 * radiation.planck_u_nu(radiation.wien_peak_nu(6000.0), 6000.0))
ax.set_xlabel("frequency nu (THz)")
ax.set_ylabel("u_nu (J s m^-3)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

for temperature in (500.0, 3000.0):
    nu_v = np.array([C / 750e-9, C / 380e-9])
    classical = 8 * np.pi * K_B * temperature / (3 * C**3) * (nu_v[1]**3 - nu_v[0]**3)
    total = radiation.energy_density(temperature)
    quantum = radiation.band_fraction(temperature, 380e-9, 750e-9) * total
    print(f"{temperature:5.0f} K, visible band: classical {classical:.3e} J/m^3, "
          f"Planck {quantum:.3e} J/m^3, whole Planck spectrum {total:.3e} J/m^3")

המכפלה *היא* חוק פלאנק, עד כדי עיגול. העקומות הקלאסיות צמודות לקוונטיות בתדירות נמוכה ואז
מטפסות ללא גבול; ב-$500\ \mathrm{K}$ התאוריה הקלאסית מכניסה לתחום האור הנראה לבדו פי אלפים
יותר אנרגיה ממה שמכיל הספקטרום הממשי כולו.

## חלק 5 — שני ספקטרומים, שני שיאים

אתרו את השיא של $u_\nu$ ואת השיא של $u_\lambda$ בכוח גס — הערך הגדול ביותר על רשת צפופה —
בכמה טמפרטורות, והשוו את $\nu_{\max}\lambda_{\max}$ ל-$c$.

**הניבוי שלכם:** מהו $\nu_{\max}\lambda_{\max}/c$?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
products = []
print("     T (K)   lambda_max (nm)   nu_max (THz)   c / nu_max (nm)   nu_max lambda_max / c")
for temperature in (300.0, 1000.0, 3000.0, 5772.0, 10000.0):
    nu = np.linspace(0.2, 3.0, 200_001) * radiation.wien_peak_nu(temperature)
    lam = np.linspace(0.2, 3.0, 200_001) * radiation.wien_peak_lambda(temperature)
    nu_peak = nu[np.argmax(radiation.planck_u_nu(nu, temperature))]
    lam_peak = lam[np.argmax(radiation.planck_u_lambda(lam, temperature))]
    products.append(nu_peak * lam_peak / C)
    print(f"{temperature:10.0f} {lam_peak * 1e9:17.1f} {nu_peak / 1e12:14.2f} "
          f"{C / nu_peak * 1e9:17.1f} {products[-1]:22.4f}")

sun = 5772.0
lam = np.linspace(100e-9, 3000e-9, 1000)
fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
nu = C / lam
left.plot(nu / 1e12, radiation.planck_u_nu(nu, sun), color="#2563eb")
left.axvline(radiation.wien_peak_nu(sun) / 1e12, color="#2563eb", label="own peak")
left.axvline(C / radiation.wien_peak_lambda(sun) / 1e12, color="#dc2626", ls="--",
             label="the other panel's peak")
left.set_xlabel("frequency (THz)")
left.set_ylabel("u_nu (J s m^-3)")
left.legend()
right.plot(lam * 1e9, radiation.planck_u_lambda(lam, sun), color="#dc2626")
right.axvline(radiation.wien_peak_lambda(sun) * 1e9, color="#dc2626", label="own peak")
right.axvline(C / radiation.wien_peak_nu(sun) * 1e9, color="#2563eb", ls="--",
              label="the other panel's peak")
right.set_xlabel("wavelength (nm)")
right.set_ylabel("u_lambda (J m^-4)")
right.legend()
for ax in (left, right):
    ax.set_title("5772 K")
plt.tight_layout()
plt.show()

היחס הוא $0.568$ בכל טמפרטורה: לפוטון שבשיא התדירות יש אורך גל הגדול פי $1.76$
מ-$\lambda_{\max}$. בטמפרטורה של השמש שיא אורך הגל ירוק ושיא התדירות אינפרה-אדום. שני הגרפים
נכונים. "השיא" הוא תכונה של הגרף, מפני שצפיפות ספקטרלית היא צפיפות.

## חלק 6 — חוק סטפן–בולצמן באינטגרציה נומרית

בצעו אינטגרציה נומרית של $u_\nu$, בלי שום הסתמכות על $\pi^4/15$, והשוו ל-$aT^4$. אחר כך חצו
שוב ושוב את מרווח הרשת ועקבו אחר השגיאה; ואחר כך בצעו אינטגרציה של $u_\lambda$ במקום, והתאימו
את המעריך של $u(T)$.

In [ ]:
exact = radiation.energy_density(300.0)
previous = None
print("intervals   relative error   ratio to previous")
for n in (64, 128, 256, 512, 1024, 2048, 4096):
    error = relative_error(radiation.integrate_u_nu(300.0, n), exact)
    ratio = f"{previous / error:8.1f}" if previous else ""
    print(f"{n:9d} {error:16.2e}   {ratio}")
    previous = error

for temperature in (2.725, 300.0, 5772.0):
    by_nu = radiation.integrate_u_nu(temperature)
    by_lambda = radiation.integrate_u_lambda(temperature)
    print(f"T = {temperature:7.3f} K: per frequency {by_nu:.6e}, per wavelength {by_lambda:.6e}, "
          f"a T^4 {radiation.energy_density(temperature):.6e} J/m^3")

temps = np.geomspace(100.0, 3000.0, 9)
exponent = scaling_exponent(temps, [radiation.integrate_u_nu(t) for t in temps])
print(f"log-log slope of u(T): {exponent:.12f};  300 K -> 600 K multiplies u by "
      f"{radiation.integrate_u_nu(600.0) / radiation.integrate_u_nu(300.0):.4f}")
gas = radiation.photon_gas(300.0)
print(f"photon gas at 300 K: n = {gas.number_density:.3e} m^-3, P = {gas.pressure:.3e} Pa, "
      f"energy per photon = {gas.energy_per_photon / (K_B * 300):.4f} k_B T")

השגיאה יורדת פי שש-עשרה בכל חציה — הסדר הרביעי של כלל סימפסון — והאינטגרל מגיע ל-$aT^4$,
ומאשר את $\pi^4/15$ שהביטוי הסגור ציטט. האינטגרל על פני אורך הגל, על רשת בלתי קשורה, נותן את
אותה אנרגיה. המעריך הוא $4$.

## חלק 7 — כסף בטמפרטורה נמוכה

קובץ הנתונים מכיל את קיבול החום של כסף בלחץ קבוע מ-$1$ עד $300\ \mathrm{K}$, כפי שרוכז בטבלאות
של NIST (סמית ופיקט, 1995), לקילוגרם. לאחר המרה למול, מתחת לכ-$6\ \mathrm{K}$ הוא אמור להיות
הסכום של $\gamma T$ של אלקטרוני ההולכה ושל $\beta T^3$ של דבאי — ולכן $C/T$ המשורטט כנגד
$T^2$ אמור להיות קו ישר.

**הניבוי שלכם:** איזה שיפוע יהיה לנתונים הגולמיים בצירים לוגריתמיים כפולים בין $1$
ל-$3\ \mathrm{K}$?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
MOLAR_MASS = 0.1078682  # kg/mol
t_ag, cp_per_kg = np.genfromtxt(DATA / "16-silver-heat-capacity.csv", delimiter=",",
                                comments="#").T
c_ag = cp_per_kg * MOLAR_MASS  # J/(mol K)

low = t_ag <= 6.0
(beta, gamma), cov = np.polyfit(t_ag[low] ** 2, c_ag[low] / t_ag[low], 1, cov=True)
theta_0 = (12 * np.pi**4 * R_GAS / (5 * beta)) ** (1 / 3)
theta_0_err = theta_0 * np.sqrt(cov[0, 0]) / (3 * beta)
print(f"gamma = {gamma * 1e3:.4f} +- {np.sqrt(cov[1, 1]) * 1e3:.4f} mJ/(mol K^2)   (paper: 0.6409)")
print(f"beta  = {beta * 1e3:.5f} mJ/(mol K^4)  ->  "
      f"theta_D = {theta_0:.1f} +- {theta_0_err:.1f} K   (paper: 226.6)")

lattice = c_ag - gamma * t_ag
raw_window, lattice_window = (t_ag >= 1) & (t_ag <= 3), (t_ag >= 1) & (t_ag <= 5)
slope_raw = np.polyfit(np.log(t_ag[raw_window]), np.log(c_ag[raw_window]), 1)[0]
slope_lattice = np.polyfit(np.log(t_ag[lattice_window]), np.log(lattice[lattice_window]), 1)[0]
print(f"log-log slope, raw data, 1-3 K: {slope_raw:.2f};  "
      f"with gamma T removed, 1-5 K: {slope_lattice:.2f}")
print(f"electrons' share of C at 1 K: {gamma * 1 / c_ag[0]:.0%}")

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
left.plot(t_ag[low] ** 2, c_ag[low] / t_ag[low] * 1e3, "o", color="k")
grid = np.linspace(0, 36, 50)
left.plot(grid, (gamma + beta * grid) * 1e3, color="#2563eb")
left.set_xlabel("T^2 (K^2)")
left.set_ylabel("C / T (mJ mol^-1 K^-2)")
right.loglog(t_ag, c_ag, "o", ms=3, color="k", label="C_P, measured")
right.loglog(t_ag, lattice, "o", ms=3, mfc="none", color="#2563eb", label="minus gamma T")
right.axhline(3 * R_GAS, color="grey", ls="--", label="3R")
right.set_xlabel("T (K)")
right.set_ylabel("C (J mol^-1 K^-1)")
right.legend()
plt.tight_layout()
plt.show()

קו ישר, כפי שהובטח: נקודת החיתוך שלו היא האלקטרונים, השיפוע שלו הוא הסריג. מתחת
לכ-$2\ \mathrm{K}$ האלקטרונים נושאים את רוב קיבול החום, וחלק גדול ממנו עד $4\ \mathrm{K}$, ולכן
השיפוע הגולמי רחוק מ-$3$ — אופן הכישלון של מתכת, במספרים של ממש. אחרי הסרת $\gamma T$ השיפוע
הוא $3$.

## חלק 8 — איינשטיין מול דבאי

התאימו את הפרמטר היחיד של כל מודל לקיבול החום של הסריג בין $4$ ל-$100\ \mathrm{K}$, על פי
שאריות יחסיות, והתבוננו בשאריות. אחר כך שאלו את הנוסחה של דבאי, נקודה אחר נקודה, איזה
$\theta_D$ דורשת כל מדידה.

**הניבוי שלכם:** איזה מודל ייכשל, והיכן?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
window = (t_ag >= 4) & (t_ag <= 100)
debye = radiation.fit_debye_temperature(t_ag[window], lattice[window], N_A)
einstein = radiation.fit_einstein_temperature(t_ag[window], lattice[window], N_A)
for name, fit in (("Debye:   ", debye), ("Einstein:", einstein)):
    print(f"{name} theta = {fit.value:6.1f} +- {fit.sigma:.1f} K, "
          f"rms residual {fit.rms_residual:.1%}")


def running_theta(temperature, heat_capacity):
    # The theta_D for which the Debye formula reproduces one measured point (bisection in log).
    low_theta, high_theta = 10.0, 3000.0
    for _ in range(60):
        mid = np.sqrt(low_theta * high_theta)
        if radiation.debye_heat_capacity(temperature, mid, N_A) > heat_capacity:
            low_theta = mid
        else:
            high_theta = mid
    return mid


running = {t: running_theta(t, c) for t, c in zip(t_ag, lattice, strict=True) if 2 <= t <= 120}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
grid = np.geomspace(1, 300, 300)
axes[0].loglog(t_ag, lattice, "o", ms=3, color="k", label="silver, lattice")
axes[0].loglog(grid, radiation.debye_heat_capacity(grid, debye.value, N_A), color="#2563eb",
               label=f"Debye {debye.value:.0f} K")
axes[0].loglog(grid, radiation.einstein_heat_capacity(grid, einstein.value, N_A), color="#dc2626",
               label=f"Einstein {einstein.value:.0f} K")
axes[0].set_ylim(1e-4, 40)
axes[0].set_xlabel("T (K)")
axes[0].set_ylabel("C (J mol^-1 K^-1)")
axes[0].legend()
axes[1].semilogx(t_ag[window], debye.residuals, "o-", color="#2563eb", label="Debye")
axes[1].semilogx(t_ag[window], einstein.residuals, "o-", color="#dc2626", label="Einstein")
axes[1].axhline(0, color="grey")
axes[1].set_xlabel("T (K)")
axes[1].set_ylabel("model / data - 1")
axes[1].legend()
axes[2].plot(list(running), list(running.values()), "o-", color="#2563eb")
axes[2].set_xlabel("T (K)")
axes[2].set_ylabel("running theta_D (K)")
plt.tight_layout()
plt.show()

העקומה של איינשטיין קורסת מעריכית מתחת לכ-$20\ \mathrm{K}$, במקום שבו זו של הכסף אינה קורסת:
תדירות אחת אינה יכולה להיות זולה ויקרה בבת אחת. דבאי עוקב אחר הנתונים בתוך כ-$10\%$ — אבל
ה-$\theta_D$ הרץ נודד בין כ-$210$ ל-$226\ \mathrm{K}$. קריקטורה בעלת פרמטר אחד של ספקטרום
הפונונים שורדת את המבחן רק אם מותר לפרמטר שלה להיסחף.

## חלק 9 — קרינת הרקע הקוסמית

קובץ הנתונים השני הוא הספקטרום של קרינת הרקע הקוסמית כפי שנמדד על ידי המכשיר FIRAS שעל
COBE, מתוך ארכיון LAMBDA של NASA. התאימו את הטמפרטורה של עקומת פלאנק, ברדיאנס
$B_\nu = (c/4\pi)\,u_\nu$, והתבוננו במה שנותר.

**הניבוי שלכם:** הנתונים משורטטים ליחידת תדירות, ומגיעים לשיא ליד $5.4\ \mathrm{cm^{-1}}$,
אורך גל של כ-$1.85\ \mathrm{mm}$. היכן יגיע לשיא אותו ספקטרום, כשהוא מבוטא ליחידת *אורך גל*?

*(רשמו כאן לפני הרצת התא הבא)*

In [ ]:
wavenumber, intensity, residual, sigma_k, _ = np.genfromtxt(
    DATA / "16-cmb-firas-monopole.csv", delimiter=",", comments="#").T
nu_cmb = 100 * C * wavenumber                 # Hz
radiance = intensity * 1e-20                  # W m^-2 Hz^-1 sr^-1
sigma = sigma_k * 1e-3 * 1e-20

cmb = radiation.fit_blackbody_temperature(nu_cmb, radiance, sigma)
print(f"T = {cmb.value:.6f} +- {cmb.sigma:.6f} K (statistical);  chi^2 = {cmb.chi_squared:.1f} "
      f"for {cmb.degrees_of_freedom} degrees of freedom")
per_lambda = radiance * nu_cmb**2 / C          # B_lambda = B_nu |d nu / d lambda|
print(f"data peak per unit frequency: {wavenumber[np.argmax(radiance)]:.2f} cm^-1 "
      f"({10 / wavenumber[np.argmax(radiance)]:.2f} mm);  per unit wavelength: "
      f"{10 / wavenumber[np.argmax(per_lambda)]:.2f} mm")

fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                                  gridspec_kw={"height_ratios": [2, 1]})
fine = np.linspace(nu_cmb.min(), nu_cmb.max(), 400)
top.plot(fine / (100 * C), radiation.planck_radiance_nu(fine, cmb.value) * 1e20, color="#2563eb",
         label=f"Planck, {cmb.value:.4f} K")
top.errorbar(wavenumber, intensity, yerr=sigma_k * 1e-3, fmt="o", ms=3, color="k", label="FIRAS")
top.set_ylabel("intensity (MJy / sr)")
top.legend()
bottom.errorbar(wavenumber, cmb.residuals, yerr=1.0, fmt="o", ms=3, color="k")
bottom.axhline(0, color="#2563eb")
bottom.set_xlabel("wavenumber (cm^-1)")
bottom.set_ylabel("residual / sigma")
plt.tight_layout()
plt.show()

עקומת פלאנק מתאימה ל-43 הנקודות עם $\chi^2$ קרוב למספר דרגות החופש: הצורה נכונה, עד כדי חלק
אחד בעשרת אלפים מהשיא. השגיאה הסטטיסטית ב-$T$ זעירה, והיא אינה כל הסיפור. הארכיון בנה את
הספקטרום הזה כעקומת פלאנק של $2.725\ \mathrm{K}$ ועוד השאריות הנמדדות, כך שההתאמה משחזרת את
הטמפרטורה שהשאריות נלקחו ביחס אליה; הטמפרטורה המוחלטת מוגבלת על ידי הכיול של FIRAS, והערך
המפורסם הוא $2.72548 \pm 0.00057\ \mathrm{K}$, וה-$\chi^2$ מתייחס לשגיאות המפורסמות כבלתי
תלויות, והן אינן בדיוק כאלה. והשיא, שוב, תלוי בציר: הנקודות הנמדדות מגיעות לשיא
ב-$5.45\ \mathrm{cm^{-1}}$, אורך גל של $1.83\ \mathrm{mm}$, ליחידת תדירות, וב-$1.05\ \mathrm{mm}$
ליחידת אורך גל (העקומה המותאמת: $1.87$ ו-$1.06\ \mathrm{mm}$; $43$ תדירויות הן רשת גסה לקריאת
שיא ממנה).

### מדידה

צטטו שתי תוצאות, כל אחת עם אי-הוודאות שלה ועם ההנחות שהיא נשענת עליהן:

- ה-$\theta_D$ של כסף מהקו של הטמפרטורות הנמוכות בחלק 7, מול $226.6\ \mathrm{K}$ (סמית ופיקט)
  — וציינו כמה הוא זז אם החלון מסתיים ב-$8\ \mathrm{K}$ במקום ב-$6$.
- ה-$T_{\mathrm{CMB}}$ מחלק 9, מול $2.725\ \mathrm{K}$ — וציינו איזו שגיאה שולטת.

## חלק 10 — הבדיקות האוטומטיות

In [ ]:
# 1. The occupation is module 12's sum.
levels = 4e-21 * np.arange(4000)
assert relative_error(radiation.mode_occupation(4e-21, 300.0),
                      partition.level_sums(levels, 300.0).energy / 4e-21) < 1e-10

# 2. Counting modes approaches the smooth formula, from below.
assert 0 < shortfalls[320] < 0.01 and shortfalls[320] < shortfalls[40] / 4

# 3. Two peaks, one ratio.
assert all(abs(p - radiation.DUAL_PEAK_RATIO) < 1e-3 for p in products)

# 4. T^4, and the same energy in either variable.
assert abs(exponent - 4.0) < 1e-6
assert relative_error(radiation.integrate_u_lambda(300.0), radiation.integrate_u_nu(300.0)) < 1e-8

# 5. Silver: the electrons and the lattice, from the low-temperature line.
assert 0.6 < gamma * 1e3 < 0.7
assert 220 < theta_0 < 232
assert 2.8 < slope_lattice < 3.2 and slope_raw < 2.2

# 6. The data prefer Debye, by far.
assert debye.rms_residual < 0.15 < 0.3 < einstein.rms_residual
assert 200 < debye.value < 235

# 7. The microwave background is a blackbody near 2.725 K.
assert abs(cmb.value - 2.725) < 0.001
assert cmb.chi_squared / cmb.degrees_of_freedom < 2.0
print("all checks passed")

## חלק 11 — חקרו בעצמכם

שני כלי חקירה; כוונו את הפקדים ולחצו על **Run Interact**.

**כלי החקירה של הגוף השחור.** הספקטרום בטמפרטורה שתבחרו, ליחידת תדירות וליחידת אורך גל זה
לצד זה, כל אחד עם השיא שלו (בקו רציף) ועם השיא של הגרף השני מועבר אליו (בקו מקווקו). דברים
לנסות: $300\ \mathrm{K}$ (חדר), $2800\ \mathrm{K}$ (חוט להט של נורה), $5772\ \mathrm{K}$ (השמש),
$10\,000\ \mathrm{K}$ (כוכב חם); העקומה הקלאסית בכל אחת מהן; צירים לוגריתמיים, שמראים את כל
הספקטרום בבת אחת.

In [ ]:
import ipywidgets as widgets


def blackbody(temperature=5772.0, classical=False, log_axes=False):
    nu_peak, lam_peak = radiation.wien_peak_nu(temperature), radiation.wien_peak_lambda(temperature)
    nu = np.linspace(0.01, 6.0, 1200) * nu_peak
    lam = np.linspace(0.05, 8.0, 1200) * lam_peak
    fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
    left.plot(nu, radiation.planck_u_nu(nu, temperature), color="#2563eb", lw=2)
    right.plot(lam * 1e9, radiation.planck_u_lambda(lam, temperature), color="#dc2626", lw=2)
    if classical:
        left.plot(nu, radiation.rayleigh_jeans_u_nu(nu, temperature), "--", color="grey")
        right.plot(lam * 1e9, radiation.rayleigh_jeans_u_nu(C / lam, temperature) * C / lam**2,
                   "--", color="grey")
    left.axvline(nu_peak, color="#2563eb")
    left.axvline(C / lam_peak, color="#dc2626", ls="--")
    right.axvline(lam_peak * 1e9, color="#dc2626")
    right.axvline(C / nu_peak * 1e9, color="#2563eb", ls="--")
    for ax in (left, right):
        if log_axes:
            ax.set_xscale("log")
            ax.set_yscale("log")
    if not log_axes:
        left.set_ylim(0, 1.3 * radiation.planck_u_nu(nu_peak, temperature))
        right.set_ylim(0, 1.3 * radiation.planck_u_lambda(lam_peak, temperature))
    left.set_xlabel("frequency (Hz)")
    left.set_ylabel("u_nu (J s m^-3)")
    right.set_xlabel("wavelength (nm)")
    right.set_ylabel("u_lambda (J m^-4)")
    plt.tight_layout()
    plt.show()
    visible = radiation.band_fraction(temperature, 380e-9, 750e-9)
    print(f"T = {temperature:.0f} K: lambda_max = {lam_peak * 1e9:.0f} nm; "
          f"nu_max = {nu_peak / 1e12:.1f} THz (the frequency of {C / nu_peak * 1e9:.0f} nm light);")
    print(f"u = {radiation.energy_density(temperature):.3e} J/m^3; visible fraction {visible:.2%}")


widgets.interact_manual(
    blackbody,
    temperature=widgets.FloatLogSlider(value=5772.0, base=10, min=2, max=4, step=0.01),
    classical=widgets.Checkbox(value=False, description="classical curve"),
    log_axes=widgets.Checkbox(value=False, description="log axes"),
)

**כלי החקירה של קיבול החום.** קיבול החום של הסריג של כסף מול העקומות של איינשטיין ושל דבאי
עבור הטמפרטורות שתבחרו, ומתחתיו השאריות היחסיות. דברים לנסות: גרמו לאיינשטיין להתאים ליד
$100\ \mathrm{K}$, ואז עברו לצירים לוגריתמיים כפולים והתבוננו ב-$5\ \mathrm{K}$; מצאו את
ה-$\theta_D$ הטוב ביותר מתחת ל-$10\ \mathrm{K}$, ואז את הטוב ביותר ליד $25\ \mathrm{K}$.

In [ ]:
def solids(theta_e=160.0, theta_d=220.0, log_log=True):
    grid = np.geomspace(1.0, 300.0, 300)
    fig, (top, bottom) = plt.subplots(2, 1, figsize=(8, 6), sharex=True,
                                      gridspec_kw={"height_ratios": [2, 1]})
    top.plot(t_ag, lattice / (3 * R_GAS), "o", ms=3, color="k", label="silver (lattice)")
    top.plot(grid, radiation.einstein_heat_capacity(grid, theta_e, N_A) / (3 * R_GAS),
             color="#dc2626", label=f"Einstein, {theta_e:.0f} K")
    top.plot(grid, radiation.debye_heat_capacity(grid, theta_d, N_A) / (3 * R_GAS),
             color="#2563eb", label=f"Debye, {theta_d:.0f} K")
    top.set_ylabel("C / 3R")
    top.legend()
    bottom.plot(t_ag, radiation.einstein_heat_capacity(t_ag, theta_e, N_A) / lattice - 1, "o-",
                ms=3, color="#dc2626")
    bottom.plot(t_ag, radiation.debye_heat_capacity(t_ag, theta_d, N_A) / lattice - 1, "o-",
                ms=3, color="#2563eb")
    bottom.axhline(0, color="grey")
    bottom.set_ylim(-1.05, 1.0)
    bottom.set_ylabel("model / data - 1")
    bottom.set_xlabel("T (K)")
    if log_log:
        top.set_xscale("log")
        top.set_yscale("log")
        top.set_ylim(1e-6, 2)
    plt.tight_layout()
    plt.show()


widgets.interact_manual(
    solids,
    theta_e=widgets.FloatSlider(value=160.0, min=50.0, max=400.0, step=1.0),
    theta_d=widgets.FloatSlider(value=220.0, min=50.0, max=400.0, step=1.0),
    log_log=widgets.Checkbox(value=True, description="log-log"),
)

## בחנו את הבנתכם

הריצו את התא שלהלן לחידון בבדיקה אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "16-radiation-solids.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שתלכו

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהנמקה שלכם?
2. בחלק 5 שני הגרפים הראו את אותה קרינה. הסבירו במילים שלכם מדוע הם מגיעים לשיא בפוטונים
   שונים, ומה *כן* זהה בשניהם.
3. המדידה שלכם של $\theta_D$ מצטטת אי-ודאות. מנו כל הנחה שהמדידה נשענת עליה ושאי-הוודאות
   *אינה* כוללת.
4. קיבול החום של כסף ב-$1\ \mathrm{K}$ הוא ברובו אלקטרוני. מדוע אין בכך סתירה לטענה שקיבול
   החום של הסריג יורד כמו $T^3$?

**התשובות שלכם:**

1.
2.
3.
4.